# Análisis del comportamiento de una aplicación bajo diferentes escenarios de despliegue

**Proyecto final · Principios y prácticas de Desarrollo de Software Orientado a Objetos**
**Autores:** Angela Lozano (`alalo10`) y Juan José Rangel García
**Repositorio:** https://github.com/alalo10/Proyecto_kubernetes_k8s

---

## 1. Introducción y objetivos

Este notebook documenta y analiza el despliegue de **una misma aplicación** (gestión de citas médicas) en distintos escenarios:

| Escenario | Descripción |
|---|---|
| **Docker Compose** | API + PostgreSQL en una sola máquina (distinta a la que genera la carga). |
| **Kubernetes 1 nodo** | K3s con un nodo, variando la API a 1, 2 y 3 réplicas. |
| **Kubernetes 2 nodos** | K3s con dos nodos, 1, 2 y 3 réplicas. |
| **Kubernetes 3 nodos** | K3s con tres nodos, 1, 2 y 3 réplicas. |

En cada escenario se genera carga con **Apache JMeter** y se miden el **tiempo medio de respuesta** y el **throughput** (peticiones por segundo). El objetivo es entender cómo el entorno de despliegue (número de réplicas y de nodos) afecta a la aplicación y qué podemos concluir sobre su **escalabilidad horizontal** y la eficiencia en el uso de recursos.

**Cómo está organizado el notebook**

1. Introducción · 2. Aplicación · 3. Configuración de despliegue · 4. Monitorización · 5. Metodología de carga · 6. Datos · 7. Gráficas · 8. Análisis comparativo · 9. Conclusiones · 10. Recomendaciones · 11. Cómo reproducir

## 2. Descripción de la aplicación

**API de Citas Médicas** (Python + FastAPI + PostgreSQL). Es un backend con lógica de negocio, accesible por API REST y con persistencia en base de datos relacional, que es lo que exige el proyecto.

**Entidades** (cada una es una clase de SQLAlchemy, es decir, una tabla):

- `Paciente`: nombre, documento (único), teléfono, correo.
- `Medico`: nombre, especialidad.
- `Cita`: paciente, médico, fecha y hora, motivo y estado (`programada`, `cancelada`, `completada`).

**Endpoints principales**

| Método | Ruta | Función |
|---|---|---|
| GET | `/citas` | Lista con filtros (médico, paciente, estado, rango de fechas) y paginación (`skip`, `limit`). |
| GET | `/citas/{id}` | Consulta una cita. |
| POST | `/citas` | Crea una cita (valida que existan paciente y médico). |
| PUT / DELETE | `/citas/{id}` | Actualiza o elimina. |
| GET/POST/PUT/DELETE | `/pacientes`, `/medicos` | CRUD equivalente. |
| GET | `/health` | Comprobación de salud (hace un `SELECT 1`); la usará Kubernetes. |

**Lógica de negocio relevante**

- Un médico **no puede tener dos citas en la misma fecha y hora**. Se valida con una consulta previa (mensaje claro, HTTP 409) y, como respaldo ante peticiones simultáneas, con una restricción `UNIQUE (medico_id, fecha_hora)` en la base de datos.
- Los listados son siempre paginados (máximo 500 filas); así una petición `GET` no devuelve las 20.000 citas de golpe y las mediciones no se distorsionan.

**Decisiones de diseño con efecto en el rendimiento**

- Índices en las columnas por las que se filtra (`medico_id`, `paciente_id`, `fecha_hora`, `estado`).
- Pool de conexiones a PostgreSQL (`pool_size=10`, `max_overflow=20`) por proceso.
- La API corre con **un único proceso de uvicorn** por contenedor. Cada réplica aprovecha, en la práctica, aproximadamente un núcleo de CPU. Esto es relevante para interpretar el efecto de añadir réplicas.
- La URL de la base de datos se lee de la variable de entorno `DATABASE_URL`, lo que permite usar **exactamente la misma imagen** (`citas-api:1.0`) en Docker Compose y en Kubernetes.

**Datos de prueba (más de 10.000 registros, requisito del proyecto):** el script `app/scripts/seed.py` carga **50 médicos, 5.000 pacientes y 20.000 citas = 25.050 registros**, con semilla fija para que los datos sean idénticos en todas las máquinas.

In [ ]:
import os, re, glob, subprocess

REPO_URL = "https://github.com/alalo10/Proyecto_kubernetes_k8s.git"
REPO_DIR = "Proyecto_kubernetes_k8s"

if os.getenv("DATOS_DIR") is None:
    # Uso normal (Colab o local): se clona el repositorio y se leen los resultados desde él
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--quiet"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--quiet", REPO_URL], check=True)
    RAIZ = REPO_DIR
    DATOS_DIR = os.path.join(RAIZ, "jmeter", "resultados")
else:
    # Modo de pruebas del propio notebook (ruta indicada por variable de entorno)
    RAIZ = os.getenv("RAIZ_REPO", ".")
    DATOS_DIR = os.environ["DATOS_DIR"]

print("Raíz del repositorio :", RAIZ)
print("Carpeta de resultados:", DATOS_DIR)
print("Archivos .jtl encontrados:", len(glob.glob(os.path.join(DATOS_DIR, "*.jtl"))))

In [ ]:
def mostrar(ruta_relativa):
    # Imprime un archivo del repositorio (si existe) para documentar la configuración
    p = os.path.join(RAIZ, ruta_relativa)
    print("=" * 70)
    print("#", ruta_relativa)
    print("=" * 70)
    if os.path.exists(p):
        print(open(p, encoding="utf-8").read())
    else:
        print("(todavía no está en el repositorio)")

for f in ["app/models.py", "app/routers/citas.py"]:
    mostrar(f)

## 3. Configuración de despliegue por escenario

### 3.1 Infraestructura utilizada

| Rol | Máquina | Recursos |
|---|---|---|
| Servidor de la app en la Fase 1 y **control plane** de K3s | Portátil de Angela (Zorin OS) | 16 GB de RAM |
| Nodo 2 del clúster | VM `agente1` (Ubuntu), proporcionada por el profesor | 1 CPU, 3,8 GB de RAM |
| Nodo 3 del clúster | VM `GP19` (Ubuntu), proporcionada por el profesor | 1 CPU, 3,8 GB de RAM |
| **Generador de carga (JMeter)** — siempre | Portátil de Rangel (Windows) | 24 GB de RAM · JMeter 5.6.3 · Java 26 |

Los clústeres se arman así: **1 nodo** = portátil Zorin; **2 nodos** = Zorin + `agente1`; **3 nodos** = Zorin + `agente1` + `GP19`.

> **Importante para interpretar los resultados:** los nodos **no son homogéneos** (un portátil frente a dos VMs de 1 CPU) y las VMs son compartidas con contenedores de otras prácticas del curso. Además, todo el tráfico pasa por una red Wi-Fi/institucional con latencia variable (en los pings vimos desde ~3 ms hasta más de 100 ms). Estos factores se discuten en las conclusiones.

### 3.2 Fase 1 · Docker Compose

`docker/docker-compose.yml` levanta dos servicios: `db` (PostgreSQL 16, con *healthcheck* y volumen persistente) y `api` (nuestra imagen), que espera a que la base esté sana. Solo el puerto 8000 de la API se expone; la base de datos queda interna.

In [ ]:
for f in ["app/Dockerfile", "docker/docker-compose.yml"]:
    mostrar(f)

### 3.3 Fases 2, 3 y 4 · Kubernetes (K3s) con 1, 2 y 3 nodos

Se usa **K3s** (Kubernetes ligero) con el control plane en el portátil Zorin y las VMs como *workers*. En cada tamaño de clúster se prueban **1, 2 y 3 réplicas de la API**, con la base de datos en un único pod. Diseño previsto: fijar PostgreSQL al nodo Zorin (`nodeSelector`) para que no caiga en una VM de 1 CPU, de modo que lo que varía entre pruebas sean las réplicas de la API y no la base de datos. Los manifiestos (Deployments, Services, etc.) están en la carpeta `k8s/` del repositorio.

In [ ]:
manifiestos = sorted(glob.glob(os.path.join(RAIZ, "k8s", "*.y*ml")))
if manifiestos:
    for p in manifiestos:
        mostrar(os.path.relpath(p, RAIZ))
else:
    print("Los manifiestos de Kubernetes todavía no están en el repositorio (carpeta k8s/).")

### 3.4 Registro de decisiones y contratiempos del proceso

- **Línea base limpia.** El portátil Zorin tenía un `k3s-agent` de un intento anterior, apuntando a un servidor que ya no existía. Se desinstaló antes de medir la Fase 1, porque K3s en segundo plano consume CPU y RAM y alteraría el punto de comparación.
- **K3s temporal en una VM.** Había también un servidor K3s recién creado en la VM `GP19`; se eliminó para dejar el control plane en el portátil (más potente).
- **Calentamiento.** La primera prueba de 10 usuarios mostró que los primeros segundos son más lentos (caches frías); por eso cada escenario incluye un calentamiento y se descartan los primeros segundos de cada medición.

## 4. Monitorización (opcional)

No se desplegó Prometheus/Grafana. Como alternativa ligera, se observó el consumo de los contenedores con `docker stats` durante algunas pruebas de la Fase 1. Son observaciones **manuales y puntuales** (la muestra es de ~1–2 s), útiles como orientación pero no como medición rigurosa.

| Prueba | Observación (`docker stats`, contenedor `citas-api`) |
|---|---|
| Compose · 10 usuarios · 30 s | Pico de ~62 % de CPU durante ~1 s; después ~34 %. Muy por debajo de un núcleo completo, lo que sugiere que con 10 usuarios la API todavía no se satura. |

*Ver la sección 11 para ampliar esta tabla con las observaciones de los demás escenarios.*

## 5. Metodología de generación de carga (JMeter)

Plan de prueba: `jmeter/plan_citas.jmx`. Es **un único plan parametrizable** (`host`, `puerto`, `usuarios`, `rampa`, `duracion`), así que exactamente la misma prueba se ejecuta en todos los escenarios y la comparación es justa.

**Mezcla de peticiones** (controlador aleatorio con 10 ranuras):

| Petición | Peso |
|---|---|
| `GET /citas?limit=20` (listado) | 30 % |
| `GET /citas/{id}` (id aleatorio entre 1 y 20.000) | 30 % |
| `GET /citas?medico_id={n}&estado=programada&limit=20` (filtrado) | 20 % |
| `POST /citas` (escritura) | 20 % |

Los `POST` usan paciente y médico aleatorios y una fecha aleatoria entre los años 2030 y 9999 para no chocar con la regla de no solapar citas ni con los datos del seed (colisión muy improbable).

**Parámetros de las pruebas**

| Parámetro | Valor |
|---|---|
| Usuarios simultáneos | 10, 50 y 100 |
| Duración por nivel | 60 s, con 10 s de rampa de arranque |
| Pausa entre peticiones (*think time*) | Ninguna: carga máxima para medir el límite del sistema |
| Calentamiento | 10 usuarios durante 15 s al comenzar cada escenario (no se guarda) |
| Ejecución | Modo consola (`-n`), sin interfaz gráfica, desde el portátil de Rangel |
| Repeticiones | 1 por nivel (limitación de tiempo) |
| Estado inicial | Base de datos reiniciada con `seed --reset` antes de cada escenario |

**Archivos de resultados:** `jmeter/resultados/{escenario}_u{usuarios}.jtl` (por ejemplo `compose_u50.jtl`, `k8s-2n-3r_u100.jtl`). El script `jmeter/correr_pruebas.ps1` automatiza todo el escenario.

**Prueba preliminar (Compose · 10 usuarios · 30 s)**, resumen de la consola de JMeter: 1.734 peticiones, 57,3 peticiones/s, media de 160 ms, **0 errores**. En los primeros 25 s la media fue de 203 ms (máximo 2,7 s) y en los últimos 6 s de 86 ms: **el arranque es más lento** y por eso se descartan los primeros segundos.

**Definición de las métricas**

- **Throughput (req/s):** peticiones completadas en la ventana de medición ÷ duración de la ventana.
- **Tiempo medio de respuesta (ms):** media de `elapsed` en la ventana. Se complementa con **p95** (el 95 % de las peticiones tardó menos que esto), porque la media esconde las colas.
- **Tasa de error (%):** peticiones con `success = false` (códigos HTTP 4xx/5xx o fallos de conexión).
- **Ventana de medición:** se descartan los primeros `DESCARTE_SEG` segundos de cada archivo (rampa de 10 s + 5 s de margen).

## 6. Carga y procesamiento de los datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

AZULES = {1: "#9ecae1", 2: "#4292c6", 3: "#08519c"}   # 1, 2 y 3 réplicas
COLOR_COMPOSE = "#6b7280"
NIVELES = [10, 50, 100]
DESCARTE_SEG = 15            # rampa (10 s) + margen de calentamiento (5 s)
ESCENARIOS = ["compose"] + [f"k8s-{n}n-{r}r" for n in (1, 2, 3) for r in (1, 2, 3)]

patron = re.compile(r"^(?P<esc>compose|k8s-(?P<nodos>[123])n-(?P<reps>[123])r)_u(?P<usuarios>\d+)\.jtl$")

def etiqueta(nodos, reps):
    return "Compose" if nodos == 0 else f"K8s {nodos}N-{reps}R"

def cargar_jtl(ruta):
    df = pd.read_csv(ruta)
    df["success"] = df["success"].astype(str).str.lower().eq("true")
    df["t"] = (df["timeStamp"] - df["timeStamp"].min()) / 1000.0   # segundos desde el inicio
    return df

crudos, filas, filas_label = {}, [], []
for ruta in sorted(glob.glob(os.path.join(DATOS_DIR, "*.jtl"))):
    nombre = os.path.basename(ruta)
    m = patron.match(nombre)
    if not m:
        print("Ignorado (nombre fuera de la convención):", nombre)
        continue
    esc, nodos, reps, usu = m["esc"], int(m["nodos"] or 0), int(m["reps"] or 0), int(m["usuarios"])
    df = cargar_jtl(ruta)
    crudos[(esc, usu)] = df
    v = df[df["t"] >= DESCARTE_SEG]
    if len(v) < 5:
        print("Muy pocos datos tras descartar el calentamiento:", nombre)
        continue
    ventana = max(v["t"].max() - DESCARTE_SEG, 1e-9)
    filas.append(dict(
        escenario=esc, etiqueta=etiqueta(nodos, reps), nodos=nodos, replicas=reps, usuarios=usu,
        n_total=len(df), n_medido=len(v), ventana_s=ventana,
        throughput_rps=len(v) / ventana,
        media_ms=v["elapsed"].mean(), p50_ms=v["elapsed"].quantile(0.50),
        p95_ms=v["elapsed"].quantile(0.95), p99_ms=v["elapsed"].quantile(0.99),
        max_ms=v["elapsed"].max(), error_pct=100 * (~v["success"]).mean(),
    ))
    for lab, g in v.groupby("label"):
        filas_label.append(dict(escenario=esc, etiqueta=etiqueta(nodos, reps), nodos=nodos, replicas=reps,
                                usuarios=usu, label=lab, n=len(g), media_ms=g["elapsed"].mean(),
                                p95_ms=g["elapsed"].quantile(0.95), error_pct=100 * (~g["success"]).mean()))

resumen = pd.DataFrame(filas)
por_label = pd.DataFrame(filas_label)
HAY_DATOS = not resumen.empty
if HAY_DATOS:
    resumen = resumen.sort_values(["nodos", "replicas", "usuarios"]).reset_index(drop=True)
    print(f"Cargadas {len(resumen)} pruebas ({int(resumen['n_medido'].sum()):,} peticiones medidas).")
else:
    print("Todavía no hay resultados .jtl en", DATOS_DIR)

**Avance de las mediciones** (✔ = prueba disponible, · = pendiente):

In [ ]:
cobertura = pd.DataFrame(
    {f"{u} usuarios": ["✔" if (e, u) in crudos else "·" for e in ESCENARIOS] for u in NIVELES},
    index=ESCENARIOS,
)
display(cobertura)

### Tabla resumen de todas las pruebas

In [ ]:
if HAY_DATOS:
    cols = ["etiqueta", "usuarios", "n_medido", "throughput_rps", "media_ms", "p50_ms", "p95_ms", "p99_ms", "max_ms", "error_pct"]
    display(resumen[cols].round(1).rename(columns={
        "etiqueta": "Escenario", "usuarios": "Usuarios", "n_medido": "Peticiones", "throughput_rps": "Throughput (req/s)",
        "media_ms": "Media (ms)", "p50_ms": "p50 (ms)", "p95_ms": "p95 (ms)", "p99_ms": "p99 (ms)",
        "max_ms": "Máx (ms)", "error_pct": "Errores (%)"}))

## 7. Resultados y visualizaciones

### 7.1 Fase 1 · Docker Compose (línea base)

In [ ]:
base = resumen[resumen["nodos"] == 0].sort_values("usuarios") if HAY_DATOS else pd.DataFrame()
if base.empty:
    print("Aún no hay mediciones de Docker Compose.")
else:
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].bar(base["usuarios"].astype(str), base["throughput_rps"], color=AZULES[3])
    ax[0].set_title("Throughput"); ax[0].set_ylabel("peticiones / s"); ax[0].set_xlabel("Usuarios simultáneos")
    ax[1].plot(base["usuarios"], base["media_ms"], marker="o", color=AZULES[2], label="Media")
    ax[1].plot(base["usuarios"], base["p95_ms"], marker="s", color=AZULES[3], label="p95")
    ax[1].set_title("Tiempo de respuesta"); ax[1].set_ylabel("ms"); ax[1].set_xlabel("Usuarios simultáneos"); ax[1].legend()
    ax[2].bar(base["usuarios"].astype(str), base["error_pct"], color="#c2410c")
    ax[2].set_title("Tasa de error"); ax[2].set_ylabel("%"); ax[2].set_xlabel("Usuarios simultáneos")
    fig.suptitle("Docker Compose · una sola máquina", y=1.03)
    plt.tight_layout(); plt.show()

**Cómo leer estas gráficas.** Con un número fijo de usuarios y sin pausa entre peticiones, el throughput es aproximadamente `usuarios ÷ tiempo de respuesta`. Si al subir de 50 a 100 usuarios el throughput **deja de crecer** y el tiempo de respuesta **sí crece**, el sistema se ha saturado (algún recurso —CPU de la API, base de datos o red— llegó a su límite).

### 7.2 Kubernetes · efecto de las réplicas y de los nodos

In [ ]:
def lineas_por_nodos(metrica, titulo, ylabel):
    if not HAY_DATOS or resumen[resumen["nodos"] > 0].empty:
        print("Aún no hay mediciones de Kubernetes."); return
    nodos_disp = sorted(resumen.loc[resumen["nodos"] > 0, "nodos"].unique())
    base = resumen[resumen["nodos"] == 0].sort_values("usuarios")
    fig, axes = plt.subplots(1, len(nodos_disp), figsize=(5.2 * len(nodos_disp), 4), sharey=True, squeeze=False)
    for ax, nd in zip(axes[0], nodos_disp):
        for r in sorted(resumen[resumen["nodos"] == nd]["replicas"].unique()):
            d = resumen[(resumen["nodos"] == nd) & (resumen["replicas"] == r)].sort_values("usuarios")
            ax.plot(d["usuarios"], d[metrica], marker="o", color=AZULES[r], label=f"{r} réplica(s)")
        if not base.empty:
            ax.plot(base["usuarios"], base[metrica], ls="--", marker="s", color=COLOR_COMPOSE, label="Compose")
        ax.set_title(f"K8s · {nd} nodo(s)"); ax.set_xlabel("Usuarios simultáneos"); ax.set_xticks(NIVELES)
    axes[0][0].set_ylabel(ylabel); axes[0][0].legend()
    fig.suptitle(titulo, y=1.03); plt.tight_layout(); plt.show()

lineas_por_nodos("throughput_rps", "Throughput según usuarios, réplicas y nodos", "peticiones / s")

In [ ]:
lineas_por_nodos("media_ms", "Tiempo medio de respuesta según usuarios, réplicas y nodos", "ms")

In [ ]:
lineas_por_nodos("p95_ms", "Tiempo de respuesta p95 según usuarios, réplicas y nodos", "ms")

### 7.3 Comparación directa a la máxima carga

In [ ]:
if HAY_DATOS:
    mx = int(resumen["usuarios"].max())
    k = resumen[(resumen["nodos"] > 0) & (resumen["usuarios"] == mx)]
    if k.empty:
        print("Aún no hay mediciones de Kubernetes.")
    else:
        p = k.pivot(index="nodos", columns="replicas", values="throughput_rps")
        fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
        p.plot(kind="bar", ax=ax[0], color=[AZULES[c] for c in p.columns], width=0.75, rot=0)
        b = resumen[(resumen["nodos"] == 0) & (resumen["usuarios"] == mx)]
        if not b.empty:
            ax[0].axhline(b["throughput_rps"].iloc[0], ls="--", color=COLOR_COMPOSE, label="Compose")
        ax[0].set_title(f"Throughput con {mx} usuarios"); ax[0].set_xlabel("Nodos del clúster"); ax[0].set_ylabel("peticiones / s")
        ax[0].set_ylim(0, max(p.values.max(), 1) * 1.3)
        ax[0].grid(axis="x", visible=False)
        ax[0].legend(title="Réplicas", loc="upper left", ncol=4)
        sns.heatmap(p, annot=True, fmt=".0f", cmap="Blues", ax=ax[1], cbar_kws={"label": "req/s"})
        ax[1].set_title(f"Mapa de calor del throughput ({mx} usuarios)"); ax[1].set_xlabel("Réplicas de la API"); ax[1].set_ylabel("Nodos")
        plt.tight_layout(); plt.show()

### 7.4 ¿Qué endpoint es el más lento?

Latencia media por tipo de petición a la máxima carga. Las consultas filtradas y los listados suelen ser más costosos que la lectura por `id`, y las escrituras dependen de la base de datos.

In [ ]:
if HAY_DATOS and not por_label.empty:
    mx = int(por_label["usuarios"].max())
    pl = por_label[por_label["usuarios"] == mx].pivot_table(index="label", columns="etiqueta", values="media_ms")
    plt.figure(figsize=(1.3 * max(len(pl.columns), 4) + 3, 3.6))
    sns.heatmap(pl, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "ms (media)"})
    plt.title(f"Latencia media por petición · {mx} usuarios"); plt.xlabel(""); plt.ylabel("")
    plt.tight_layout(); plt.show()
else:
    print("Sin datos todavía.")

### 7.5 Comportamiento en el tiempo (justificación del calentamiento)

Peticiones por segundo y latencia media a lo largo de una prueba. La zona sombreada es la que se **descarta** (rampa + calentamiento).

In [ ]:
TIMELINE = None    # p. ej. ("compose", 50); por defecto se elige una prueba disponible
if crudos:
    clave = TIMELINE or (("compose", 50) if ("compose", 50) in crudos else sorted(crudos)[0])
    df = crudos[clave]
    seg = df.groupby(df["t"].astype(int)).agg(req=("elapsed", "size"), media=("elapsed", "mean"))
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.axvspan(0, DESCARTE_SEG, color="#dbeafe", label="Descartado")
    ax1.bar(seg.index, seg["req"], color=AZULES[2], label="Peticiones/s")
    ax1.set_xlabel("Segundos desde el inicio"); ax1.set_ylabel("peticiones / s")
    ax2 = ax1.twinx(); ax2.plot(seg.index, seg["media"], color="#c2410c", label="Latencia media")
    ax2.set_ylabel("ms"); ax2.spines["right"].set_visible(True)
    ax1.set_title(f"Línea de tiempo · {clave[0]} · {clave[1]} usuarios")
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper right"); plt.tight_layout(); plt.show()
else:
    print("Sin datos todavía.")

### 7.6 Errores

In [ ]:
if HAY_DATOS:
    e = resumen[["etiqueta", "usuarios", "n_medido", "error_pct"]].copy()
    e["error_pct"] = e["error_pct"].round(2)
    display(e[e["error_pct"] > 0].rename(columns={"etiqueta": "Escenario", "usuarios": "Usuarios",
                                                  "n_medido": "Peticiones", "error_pct": "Errores (%)"})
            if (e["error_pct"] > 0).any() else "Ninguna prueba registró errores en la ventana medida.")

## 8. Análisis comparativo

El texto de abajo se **genera automáticamente** a partir de los datos cargados (así siempre coincide con las gráficas). La interpretación —el *por qué*— se escribe en la sección 9.

In [ ]:
def hallazgos():
    if not HAY_DATOS:
        return ["Todavía no hay mediciones cargadas."]
    out = []
    mx = int(resumen["usuarios"].max())
    en_mx = resumen[resumen["usuarios"] == mx].sort_values("throughput_rps", ascending=False)
    mejor = en_mx.iloc[0]
    out.append(f"Con {mx} usuarios, el mayor throughput fue {mejor['throughput_rps']:.1f} req/s en {mejor['etiqueta']} "
               f"(media {mejor['media_ms']:.0f} ms, p95 {mejor['p95_ms']:.0f} ms).")

    base = en_mx[en_mx["nodos"] == 0]
    kube = en_mx[en_mx["nodos"] > 0]
    if not base.empty and not kube.empty:
        b = base.iloc[0]["throughput_rps"]
        kb, kp = kube.iloc[0], kube.iloc[-1]
        out.append(f"El mejor escenario de Kubernetes ({kb['etiqueta']}) rindió {100 * (kb['throughput_rps'] / b - 1):+.0f}% "
                   f"respecto a Docker Compose ({b:.1f} req/s); el peor ({kp['etiqueta']}), {100 * (kp['throughput_rps'] / b - 1):+.0f}%.")

    for nd in sorted(kube["nodos"].unique()):
        s = kube[kube["nodos"] == nd].set_index("replicas")["throughput_rps"]
        if 1 in s.index and len(s) > 1:
            rmax = s.index.max()
            out.append(f"Con {nd} nodo(s), pasar de 1 a {rmax} réplicas cambió el throughput en "
                       f"{100 * (s[rmax] / s[1] - 1):+.0f}% ({s[1]:.1f} → {s[rmax]:.1f} req/s).")

    for r in sorted(kube["replicas"].unique()):
        s = kube[kube["replicas"] == r].set_index("nodos")["throughput_rps"]
        if len(s) > 1:
            n0, n1 = s.index.min(), s.index.max()
            out.append(f"Con {r} réplica(s), pasar de {n0} a {n1} nodo(s) cambió el throughput en "
                       f"{100 * (s[n1] / s[n0] - 1):+.0f}% ({s[n0]:.1f} → {s[n1]:.1f} req/s).")

    niveles = sorted(resumen["usuarios"].unique())
    if len(niveles) >= 2:
        a, z = niveles[-2], niveles[-1]
        for e, g in resumen.groupby("etiqueta"):
            s = g.set_index("usuarios")["throughput_rps"]
            if a in s.index and z in s.index and s[a] > 0 and s[z] / s[a] < 1.10:
                out.append(f"{e} parece saturado: de {a} a {z} usuarios el throughput varió solo {100 * (s[z] / s[a] - 1):+.0f}%.")

    malos = resumen[resumen["error_pct"] > 1]
    if malos.empty:
        out.append("Ninguna prueba superó el 1 % de errores.")
    else:
        out.append("Pruebas con más de 1 % de errores: " +
                   ", ".join(f"{r.etiqueta} @ {r.usuarios} usuarios ({r.error_pct:.1f} %)" for r in malos.itertuples()))
    return out

for linea in hallazgos():
    print("•", linea)

### Eficiencia de las réplicas

Si duplicar las réplicas duplicara el throughput, la eficiencia sería del 100 %. Valores bajos indican que otro recurso (base de datos, red, nodos más débiles o el propio generador de carga) limita el escalado.

In [ ]:
if HAY_DATOS and not resumen[resumen["nodos"] > 0].empty:
    mx = int(resumen["usuarios"].max())
    k = resumen[(resumen["nodos"] > 0) & (resumen["usuarios"] == mx)].copy()
    ref = k[k["replicas"] == 1].set_index("nodos")["throughput_rps"]
    k["speedup_vs_1_replica"] = k.apply(lambda f: f["throughput_rps"] / ref[f["nodos"]] if f["nodos"] in ref.index else np.nan, axis=1)
    k["eficiencia_pct"] = 100 * k["speedup_vs_1_replica"] / k["replicas"]
    display(k[["etiqueta", "throughput_rps", "speedup_vs_1_replica", "eficiencia_pct"]].round(2)
            .rename(columns={"etiqueta": "Escenario", "throughput_rps": "Throughput (req/s)",
                             "speedup_vs_1_replica": "Aceleración vs 1 réplica", "eficiencia_pct": "Eficiencia (%)"}))
else:
    print("Sin datos de Kubernetes todavía.")

## 9. Conclusiones

### 9.1 Lo que ya sabemos por el diseño y las pruebas preliminares

Estos puntos condicionan **cómo hay que leer** cualquier resultado:

1. **Nodos heterogéneos.** El portátil Zorin es mucho más potente que las VMs (1 CPU, 3,8 GB), que además comparten recursos con otras prácticas. Por eso **añadir una VM al clúster puede no mejorar el rendimiento, o incluso empeorarlo**: sería un hallazgo del análisis y no un error de configuración.
2. **Red con latencia variable.** Entre el generador de carga y la app, y entre los nodos del clúster, el tráfico pasa por Wi-Fi/red institucional. Parte del tiempo de respuesta es de red y no de la aplicación.
3. **Un proceso de API por réplica.** Cada réplica usa aproximadamente un núcleo. En la prueba preliminar con 10 usuarios la CPU de la API alcanzó un pico de ~62 % (no saturada). **Hipótesis a verificar:** con 50 y 100 usuarios la API sí se satura, y ahí es donde más deberían notarse las réplicas.
4. **Una única base de datos.** Escalar la API sin escalar PostgreSQL tiene un techo: el 20 % de las peticiones son escrituras y todas las réplicas comparten la misma base. Puede convertirse en el cuello de botella.
5. **Un solo generador de carga.** A 100 usuarios, el portátil de JMeter también podría limitar el throughput observado.
6. **Calentamiento y repeticiones.** Los primeros segundos son más lentos (caches frías), por lo que se descartan; y con **una sola repetición** por prueba no podemos calcular intervalos de confianza: diferencias pequeñas (del orden de unos pocos %) no deben interpretarse como reales.
7. **Estado de la base.** Cada `POST` añade filas. Para que los escenarios partan igual, se reinicia el seed antes de cada uno.

### 9.2 Conclusiones basadas en los datos

> ✏️ **Para completar tras las mediciones.** Guía: usa las gráficas de las secciones 7 y 8 y responde, con cifras:
>
> - **Docker Compose:** ¿a cuántos usuarios se satura? ¿Cuál es el throughput máximo y el p95 en ese punto?
> - **Réplicas (1 → 2 → 3):** ¿mejoran el throughput? ¿Con qué eficiencia? ¿Dejan de mejorar a partir de cierto número?
> - **Nodos (1 → 2 → 3):** ¿qué cambia al sumar una VM de 1 CPU? ¿Coincide con la hipótesis de los nodos heterogéneos?
> - **Compose vs Kubernetes** con los mismos recursos: ¿cuánto cuesta (o aporta) la orquestación?
> - **Cuello de botella:** ¿API, base de datos, red o generador de carga? ¿Qué evidencia lo respalda (latencia por endpoint, CPU observada, errores)?
> - **Escalabilidad horizontal:** ¿es lineal, sublineal o no escala en este entorno? ¿Por qué?

## 10. Recomendaciones

> ✏️ **Para confirmar con los datos.** Posibles recomendaciones si el análisis las respalda:
>
> - Ejecutar **varios procesos por contenedor** (p. ej. Gunicorn con workers de uvicorn) si la CPU de una réplica se satura antes que la base de datos.
> - Añadir un **balanceador de conexiones** (PgBouncer) o ajustar el pool si las réplicas multiplican las conexiones a PostgreSQL.
> - Revisar **índices compuestos** si el filtro por médico y estado resulta el endpoint más lento.
> - Usar **nodos homogéneos** (o `nodeSelector`/afinidad según capacidad) para evitar que las réplicas caigan en nodos débiles.
> - Repetir cada prueba **3 veces o más** y reportar medias con intervalos de confianza.
> - Si hay tiempo, desplegar **Prometheus + Grafana** para correlacionar CPU/RAM con las métricas de JMeter.

## 11. Cómo reproducir el proyecto

1. **Clonar el repositorio:** `git clone https://github.com/alalo10/Proyecto_kubernetes_k8s.git`
2. **Fase 1 (Compose):** en `docker/`, `docker compose up -d` y cargar los datos con `docker compose exec api python -m scripts.seed`.
3. **Fases 2–4 (Kubernetes):** instalar K3s (control plane en el portátil y VMs como workers) y aplicar los manifiestos de `k8s/`.
4. **Cargar y medir:** desde otra máquina, `.\jmeter\correr_pruebas.ps1 -Escenario <nombre> -HostApi <IP>`. Nombres válidos: `compose`, `k8s-1n-1r` … `k8s-3n-3r`.
5. **Subir resultados:** `git add jmeter/resultados && git commit && git push`.
6. **Analizar:** abrir este notebook en Google Colab y ejecutar *Entorno de ejecución → Ejecutar todo*. Clona el repositorio y lee los `.jtl`.

Las instrucciones detalladas y los enlaces están también en el reporte PDF de la carpeta `report/`.